# Part I: Testing SparkDataCheck

In [2]:
from pyspark.sql import SparkSession
import spark_wrapper
from pyspark.sql.types import *
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/24 16:30:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/24 16:30:53 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
import pandas as pd
pandas_df = pd.read_csv("weekly_nfl_data.csv")

In [21]:
import importlib
importlib.reload(spark_wrapper)

<module 'spark_wrapper' from '/home/jupyter-sbenjam3@ncsu.edu/ST554_Project2/spark_wrapper.py'>

In [22]:
check = spark_wrapper.SparkDataCheck()
check.instance_from_pd(spark, pandas_df)

In [27]:
check.min_and_max().select("min_passing_yards").show()

+-----------------+
|min_passing_yards|
+-----------------+
|            -16.0|
+-----------------+



In [17]:
check.string_counts("position", "season_type").show()

+-----------+---------------+
|season_type|count(position)|
+-----------+---------------+
|       POST|           5335|
|        REG|         123538|
+-----------+---------------+



In [14]:
check.df.select("position").distinct().show(50)

+--------+
|position|
+--------+
|       K|
|      QB|
|      RB|
|      FB|
|      TE|
|      LS|
|       G|
|      WR|
|       P|
|      DB|
|      LB|
|     NaN|
|      CB|
|       T|
|      FS|
|     MLB|
|      SS|
|       C|
|     ILB|
|     OLB|
|      DT|
|      NT|
|      OG|
|      OL|
|      DE|
|      OT|
|      DL|
|       S|
|      HB|
+--------+



In [15]:
check.df.select("position_group").distinct().show(50)

+--------------+
|position_group|
+--------------+
|          SPEC|
|            OL|
|            QB|
|            RB|
|            TE|
|            WR|
|            DB|
|            LB|
|           NaN|
|            DL|
+--------------+



In [7]:
check.df.select(["player_display_name","missing_check"]).show()

26/03/23 11:09:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+--------------------+-------------+
| player_display_name|missing_check|
+--------------------+-------------+
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|Abdul-Karim al-Ja...|        false|
|      Rabih Abdullah|        false|
|      Rabih Abdullah|        false|
|      Rabih Abdullah|        false|
|      Rabih Abdullah|        false|
|         Troy Aikman|        false|
|         Troy Aikman|        false|
|         Troy Aikman|        false|
+--------------------+-------------+
only showing top 20 rows


In [10]:
numeric_cols = [col_name for col_name, col_type in check.df.dtypes if isinstance(check.df.schema[col_name].dataType, NumericType)]
numeric_cols

['season',
 'week',
 'completions',
 'attempts',
 'passing_yards',
 'passing_tds',
 'interceptions',
 'sacks',
 'sack_yards',
 'sack_fumbles',
 'sack_fumbles_lost',
 'passing_air_yards',
 'passing_yards_after_catch',
 'passing_first_downs',
 'passing_epa',
 'passing_2pt_conversions',
 'pacr',
 'dakota',
 'carries',
 'rushing_yards',
 'rushing_tds',
 'rushing_fumbles',
 'rushing_fumbles_lost',
 'rushing_first_downs',
 'rushing_epa',
 'rushing_2pt_conversions',
 'receptions',
 'targets',
 'receiving_yards',
 'receiving_tds',
 'receiving_fumbles',
 'receiving_fumbles_lost',
 'receiving_air_yards',
 'receiving_yards_after_catch',
 'receiving_first_downs',
 'receiving_epa',
 'receiving_2pt_conversions',
 'racr',
 'target_share',
 'air_yards_share',
 'wopr',
 'special_teams_tds',
 'fantasy_points',
 'fantasy_points_ppr']

In [12]:
column = "season"
if types[column] != 'bigint':
    print("no")
else:
    print("yes")

yes


# Part II: Basic Analysis with Spark
In this section, we'll use the pandas-on-Spark and the SQL API to analyze data from an NFL dataset. The Pandas API on Spark allows us to easily transition from pandas to Spark and use a lot of familar Pandas commands. Spark SQL allows us to efficiently read, write, tranform and analyze data using SQL commands. This dataset contains a variety of fields including players, their positions and corresponding position groups, the teams they played on, and player statistics. Using Spark, we can navigate through this large dataset and make conclusions about what the data is saying.

## Pandas-on-Spark
We start by importing all the necessary modules including Spark and Pandas. Then we set up our spark session using `SparkSession`. 

In [1]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql.functions import *
ps.set_option('compute.fail_on_ansi_mode', False)

/opt/tljh/user/envs/pySpark3/lib/python3.9/site-packages/pyspark/pandas/__init__.py:43: UserWarning: 'PYARROW_IGNORE_TIMEZONE' environment variable was not set. It is required to set this environment variable to '1' in both driver and executor sides if you use pyarrow>=2.0.0. pandas-on-Spark will set it for you but it does not work if there is a Spark context already launched.
  warnings.warn(
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/25 10:02:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/25 10:02:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
#set up spark session
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

26/03/25 10:02:24 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


To create our Spark DataFrame, we first create a Pandas DataFrame using `read_csv()`. Then we use `from_pandas()` from `pyspark.pandas` to convert our Pandas DataFrame to a Spark DataFrame. Using `head()`, we can see the columns printed out along with their values.  

In [3]:
#create pandas DataFrame of the NFL dataset 
pdf = pd.read_csv("weekly_nfl_data.csv")
#create pyspark DataFrame from pandas DataFrame
psdf = ps.from_pandas(pdf)
#print the first five rows of the NFL dataset 
psdf.head()

26/03/25 10:02:32 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


To look at the column names of our NFL dataset, we can use `dtypes` to print out the column names along with the type of data stored in these columns. 

In [46]:
#get column names and data types of DataFrame
psdf.dtypes

player_id                       object
player_name                     object
player_display_name             object
position                        object
position_group                  object
headshot_url                    object
recent_team                     object
season                           int64
week                             int64
season_type                     object
opponent_team                   object
completions                      int64
attempts                         int64
passing_yards                  float64
passing_tds                      int64
interceptions                  float64
sacks                          float64
sack_yards                     float64
sack_fumbles                     int64
sack_fumbles_lost                int64
passing_air_yards              float64
passing_yards_after_catch      float64
passing_first_downs            float64
passing_epa                    float64
passing_2pt_conversions          int64
pacr                     

Now we want to start filtering our data. Using familiar Pandas syntax, we can subset our original DataFrame to return only **regular seasons between 2005 and 2023**, where the position is **QB**. We also subset the DataFrame to return a small selection of the columns. We use `head()` to look at only the first five rows but there are a total of 11,750 records that fall under these conditions.

In [47]:
#subset oringal DataFrame to include seasons between 2005 and 2023, regular seasons, QB positions, and certain column names 
new_df = psdf[(psdf['season'] >=2005) & (psdf['season'] <=2023) & (psdf['season_type'] == 'REG') & (psdf['position'] == 'QB')][['player_display_name','season','week','completions','attempts','passing_yards','passing_tds','interceptions']]
#calculate and print the length of the new DataFrame
print(f"The resulting subsetted DataFrame contains {len(new_df)} records.")
new_df.head()

The resulting subsetted DataFrame contains 11750 records.


,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


With our new subsetted DataFrame, we can use `groupby()` to find the sum and mean for the playing statistics of each player in each season. Using `agg()` we can apply the sum and mean operations to remaining columns in the DataFrame. Using `head()`, we can look at the resulting statistis and see, for example, that in the 2005 season Jeff Blake has an average of 27.5 passing yards and a total of 9 attempts. We can use these new sums and means to calculate a few new statistics, to calcualte the completion percentage `completion_percentage`, we create a new column that takes the summed completions and divides it by the summed attempts. We also create a variable for the touchdown/interception ratio `td_int_ratio` which takes the summed passing touchdowns and divides it by the summed interceptions. For the player Jeff Blake in the first row, we can see that there were no interceptions and therefore dividing by 0 to calculate `td_int_ratio` results in a value of `inf`. 

In [14]:
new_df_stats = new_df.groupby(['player_display_name','season']).agg(['sum', 'mean'])

In [48]:
#add completion_percentage variable
new_df_stats['completion_percentage'] = (new_df_stats['completions']['sum'])/(new_df_stats['attempts']['sum'])
#add td_int_ratio variable
new_df_stats['td_int_ratio'] =  (new_df_stats['passing_tds']['sum'])/(new_df_stats['interceptions']['sum'])
new_df_stats.head()

week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
Jeff Blake          2005     19   9.500000           8   4.000000        9   4.500000          55.0   27.500000           1  0.500000           0.0  0.000000              0.888889          inf
Daunte Culpepper    2005     31   4.428571         139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005    134   8.933333         302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Tony Banks          2005     17  17.000000          14  14.000000       25  25.000000         173.0  173.000000           1  1.000000           2.0  2.000000              0.560000     0.500000
Charlie Batch       2005     35  11.666667          23   7.666667       36  12.000000         246.0   82.000000           1  0.333333           1.0  0.333333              0.638889     1.000000

We now want to further subset and sort our DataFrame. We first filter our DataFrame to include records where the number of attempts is at least 50. The size of the DataFrame now goes to 966 records. 

In [49]:
#filter DataFrame to include attempts that are at least 50
attempts_over_50 = new_df_stats[new_df_stats['attempts']['sum'] >= 50]
#calculate and print length of the new DataFrame
print(f"There are {len(attempts_over_50)} records where the number of attempts are over 50.")
attempts_over_50.head()

There are 966 records where the number of attempts are over 50.


week            completions            attempts            passing_yards             passing_tds           interceptions           completion_percentage td_int_ratio
                            sum       mean         sum       mean      sum       mean           sum        mean         sum      mean           sum      mean                                   
player_display_name season                                                                                                                                                                      
Daunte Culpepper    2005     31   4.428571         139  19.857143      216  30.857143        1564.0  223.428571           6  0.857143          12.0  1.714286              0.643519     0.500000
Kerry Collins       2005    134   8.933333         302  20.133333      566  37.733333        3759.0  250.600000          20  1.333333          13.0  0.866667              0.533569     1.538462
Jeff Garcia         2005     69  11.500000         102  17.000000      173  28.833333         937.0  156.166667           3  0.500000           6.0  1.000000              0.589595     0.500000
Brett Favre         2005    147   9.187500         372  23.250000      607  37.937500        3881.0  242.562500          20  1.250000          29.0  1.812500              0.612850     0.689655
Todd Bouman         2005     60  12.000000          68  13.600000      122  24.400000         722.0  144.400000           2  0.400000           7.0  1.400000              0.557377     0.285714

First, we sort the results in a decsending fashion by the variable `completion_percentage` with `sort_values()` and show the first 40 rows. We see that there are a number of players that have a 100% completion percentage.

In [50]:
#sort descending by completion_percentage
descending_by_cp = new_df_stats.sort_values(by='completion_percentage', ascending=False)
#show only completions, attempts, and completion_percetange columns
descending_by_cp[['completions','attempts','completion_percentage']][:40]

completions            attempts            completion_percentage
                                   sum       mean      sum       mean                      
player_display_name season                                                                 
Anthony Wright      2006             3   1.500000        3   1.500000              1.000000
Matt Gutierrez      2007             1   0.250000        1   0.250000              1.000000
Dennis Dixon        2008             1   1.000000        1   1.000000              1.000000
Matt Gutierrez      2009             1   1.000000        1   1.000000              1.000000
Brad Smith          2009             1   0.090909        1   0.090909              1.000000
Billy Volek         2010             1   0.333333        1   0.333333              1.000000
Jordan Palmer       2010             3   3.000000        3   3.000000              1.000000
Brian Hoyer         2011             1   0.500000        1   0.500000              1.000000
Tyrod Taylor        2011             1   0.500000        1   0.500000              1.000000
Derek Anderson      2012             4   2.000000        4   2.000000              1.000000
Trent Edwards       2012             2   2.000000        2   2.000000              1.000000
Chase Daniel        2012             1   1.000000        1   1.000000              1.000000
Colt McCoy          2013             1   0.333333        1   0.333333              1.000000
Tarvaris Jackson    2014             1   1.000000        1   1.000000              1.000000
Matt Moore          2015             1   1.000000        1   1.000000              1.000000
Chase Daniel        2015             2   1.000000        2   1.000000              1.000000
Scott Tolzien       2015             1   0.500000        1   0.500000              1.000000
Ryan Nassib         2015             5   5.000000        5   5.000000              1.000000
Chase Daniel        2016             1   1.000000        1   1.000000              1.000000
Taylor Heinicke     2017             1   1.000000        1   1.000000              1.000000
Alex Tanney         2019             1   1.000000        1   1.000000              1.000000
AJ McCarron         2020             1   0.500000        1   0.500000              1.000000
Easton Stick        2020             1   1.000000        1   1.000000              1.000000
C.J. Beathard       2021             2   1.000000        2   1.000000              1.000000
Brandon Allen       2022             3   3.000000        3   3.000000              1.000000
Sean Clifford       2023             1   0.500000        1   0.500000              1.000000
Mike Glennon        2016            10  10.000000       11  11.000000              0.909091
Kyle Orton          2012             9   9.000000       10  10.000000              0.900000
Jeff Blake          2005             8   4.000000        9   4.500000              0.888889
Quinn Gray          2008             7   7.000000        8   8.000000              0.875000
Josh Johnson        2010            14   1.750000       16   2.000000              0.875000
Sean Mannion        2015             6   6.000000        7   7.000000              0.857143
Nick Mullens        2022            21   5.250000       25   6.250000              0.840000
Kellen Clemens      2015             5   2.500000        6   3.000000              0.833333
Brian Hoyer         2022             5   5.000000        6   6.000000              0.833333
Mike White          2023             5   0.833333        6   1.000000              0.833333
Nate Sudfeld        2017            19  19.000000       23  23.000000              0.826087
Landry Jones        2017            23   7.666667       28   9.333333              0.821429
Luke McCown         2015            32  16.000000       39  19.500000              0.820513
Brian Hoyer         2021             9   1.800000       11   2.200000              0.818182

Next, we sort the DataFrame in an ascending fashion by `td_int_ratio` and show the first 40 records. The results show us that there are a number of players who had seasons where that had zero touchdowns giving them a `td_int_ratio` of 0. 

In [51]:
#sort ascending by td_int_ratio
ascending_by_td = new_df_stats.sort_values(by='td_int_ratio')
#show only passing_tds, interceptions, and td_int_ratio columns
ascending_by_td[['passing_tds', 'interceptions','td_int_ratio']][:40]

passing_tds      interceptions           td_int_ratio
                                   sum mean           sum      mean             
player_display_name season                                                      
Koy Detmer          2005             0  0.0           3.0  0.600000          0.0
Jon Kitna           2005             0  0.0           2.0  0.666667          0.0
Cody Pickett        2005             0  0.0           2.0  0.666667          0.0
Philip Rivers       2005             0  0.0           1.0  0.500000          0.0
Matt Mauck          2005             0  0.0           1.0  0.500000          0.0
Aaron Rodgers       2005             0  0.0           1.0  0.333333          0.0
Brooks Bollinger    2006             0  0.0           1.0  0.500000          0.0
Brett Basanez       2006             0  0.0           1.0  1.000000          0.0
Brodie Croyle       2006             0  0.0           2.0  1.000000          0.0
Billy Volek         2007             0  0.0           1.0  0.200000          0.0
Brock Berlin        2007             0  0.0           1.0  1.000000          0.0
Charlie Frye        2007             0  0.0           1.0  1.000000          0.0
Bruce Gradkowski    2007             0  0.0           1.0  0.250000          0.0
Matt Cassel         2007             0  0.0           1.0  0.200000          0.0
Trent Green         2008             0  0.0           6.0  2.000000          0.0
Tyler Thigpen       2007             0  0.0           1.0  1.000000          0.0
Ken Dorsey          2008             0  0.0           7.0  1.750000          0.0
Andrew Walter       2008             0  0.0           3.0  1.500000          0.0
Bruce Gradkowski    2008             0  0.0           3.0  1.500000          0.0
Kellen Clemens      2008             0  0.0           1.0  0.500000          0.0
Jordan Palmer       2008             0  0.0           2.0  0.666667          0.0
Kevin Kolb          2008             0  0.0           4.0  0.666667          0.0
Mark Brunell        2009             0  0.0           1.0  0.250000          0.0
Chris Simms         2009             0  0.0           1.0  0.333333          0.0
Rex Grossman        2009             0  0.0           1.0  1.000000          0.0
Matt Leinart        2009             0  0.0           3.0  0.375000          0.0
Caleb Hanie         2009             0  0.0           1.0  0.500000          0.0
Drew Stanton        2009             0  0.0           6.0  2.000000          0.0
Troy Smith          2009             0  0.0           1.0  0.333333          0.0
Matt Flynn          2009             0  0.0           1.0  0.250000          0.0
Brian Brohm         2009             0  0.0           2.0  1.000000          0.0
Curtis Painter      2009             0  0.0           2.0  1.000000          0.0
Todd Collins        2010             0  0.0           5.0  2.500000          0.0
David Carr          2010             0  0.0           1.0  1.000000          0.0
Kyle Boller         2010             0  0.0           1.0  0.333333          0.0
Brodie Croyle       2010             0  0.0           1.0  0.500000          0.0
Dennis Dixon        2010             0  0.0           1.0  0.500000          0.0
Richard Bartel      2010             0  0.0           1.0  1.000000          0.0
Brian Brohm         2010             0  0.0           3.0  3.000000          0.0
Levi Brown          2010             0  0.0           1.0  1.000000          0.0

In the section above, we saw how to use pandas-on-Spark to look through out NFL dataset, subset and group the data, and calculate new player statistics that tell us about their performance during different seasons.

## Spark SQL
In this section, we'll switch to using Spark SQL to complete the same actions we did above with our NFL dataset. Like the previous section, we start by importing the `pyspark.sql` module and creating our spark session.

In [17]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('my_app').config("spark.sql.ansi.enabled", "false").getOrCreate()

Using `read.load()` we can convert the csv file containing our NFL data to a Spark SQL DataFrame. 

In [52]:
#load csv data to spark DataFrame
sql_df = spark.read.load("weekly_nfl_data.csv",
                     format="csv", 
                     sep=",", 
                     inferSchema="true", 
                     header="true")

In Spark SQL, `.show()` is used to print out rowns of data in the DataFrame instead of `head()` used with pandas-on-Spark. Since we have so many columns, the data doesn't output as nicely as pandas-on-Spark. We can use the `select()` to select only a few of the columns and get a nicer view of our dataset. 

In [53]:
#output first five lines of the DataFrame
sql_df.show(5)

+----------+-----------+--------------------+--------+--------------+------------+-----------+------+----+-----------+-------------+-----------+--------+-------------+-----------+-------------+-----+----------+------------+-----------------+-----------------+-------------------------+-------------------+-----------+-----------------------+----+------+-------+-------------+-----------+---------------+--------------------+-------------------+-----------+-----------------------+----------+-------+---------------+-------------+-----------------+----------------------+-------------------+---------------------------+---------------------+-------------+-------------------------+----+------------+---------------+----+-----------------+--------------+------------------+
| player_id|player_name| player_display_name|position|position_group|headshot_url|recent_team|season|week|season_type|opponent_team|completions|attempts|passing_yards|passing_tds|interceptions|sacks|sack_yards|sack_fumbles|sack_

In [54]:
#show only player_display_name, season, position, position_group, and recent_team columns
sql_df.select(['player_display_name','season','position','position_group','recent_team']).show(5)

+--------------------+------+--------+--------------+-----------+
| player_display_name|season|position|position_group|recent_team|
+--------------------+------+--------+--------------+-----------+
|Abdul-Karim al-Ja...|  1999|      RB|            RB|        MIA|
|Abdul-Karim al-Ja...|  1999|      RB|            RB|        MIA|
|Abdul-Karim al-Ja...|  1999|      RB|            RB|        MIA|
|Abdul-Karim al-Ja...|  1999|      RB|            RB|        CLE|
|Abdul-Karim al-Ja...|  1999|      RB|            RB|        CLE|
+--------------------+------+--------+--------------+-----------+
only showing top 5 rows


To report the column names in our dataset along with their data types, we can use `dtypes` as we did in the previous section. We can see that there are a total of 53 columns in this dataset.

In [55]:
#output column name and data types of DataFrame
print(f"There are {len(sql_df.dtypes)} columns in the NFL dataset.")
sql_df.dtypes

There are 53 columns in the NFL dataset.


[('player_id', 'string'),
 ('player_name', 'string'),
 ('player_display_name', 'string'),
 ('position', 'string'),
 ('position_group', 'string'),
 ('headshot_url', 'string'),
 ('recent_team', 'string'),
 ('season', 'int'),
 ('week', 'int'),
 ('season_type', 'string'),
 ('opponent_team', 'string'),
 ('completions', 'int'),
 ('attempts', 'int'),
 ('passing_yards', 'double'),
 ('passing_tds', 'int'),
 ('interceptions', 'double'),
 ('sacks', 'double'),
 ('sack_yards', 'double'),
 ('sack_fumbles', 'int'),
 ('sack_fumbles_lost', 'int'),
 ('passing_air_yards', 'double'),
 ('passing_yards_after_catch', 'double'),
 ('passing_first_downs', 'double'),
 ('passing_epa', 'double'),
 ('passing_2pt_conversions', 'int'),
 ('pacr', 'double'),
 ('dakota', 'double'),
 ('carries', 'int'),
 ('rushing_yards', 'double'),
 ('rushing_tds', 'int'),
 ('rushing_fumbles', 'double'),
 ('rushing_fumbles_lost', 'double'),
 ('rushing_first_downs', 'double'),
 ('rushing_epa', 'double'),
 ('rushing_2pt_conversions', 'int

Next, we want to look at data  for **quarterbacks (QB) from regular seasons between 2005 and 2023** and only look at a few of the columns. To subset only certain columns, we use `select()` and pass in the desired column names. We then use `filter()` to pass the conditions described above. We see that our new DataFrame has 11,750 records which matches the results when using pandas-on-Spark.

In [56]:
#subset DataFrame to include regular sesasons between 2005 and 2023 and QB positions
sql_short = sql_df.select(['player_display_name','season', 'week', 'completions','attempts','passing_yards','passing_tds','interceptions']).filter((sql_df.season.between(2005,2023)) & (sql_df.position == 'QB') & (sql_df.season_type == 'REG'))
#calculate and print the number of records in the new DataFrame
print(f"There are {sql_short.count()} records in the new DataFrame.")
sql_short.show()

There are 11750 records in the new DataFrame.
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|player_display_name|season|week|completions|attempts|passing_yards|passing_tds|interceptions|
+-------------------+------+----+-----------+--------+-------------+-----------+-------------+
|         Tony Banks|  2005|  17|         14|      25|        173.0|          1|          2.0|
|      Charlie Batch|  2005|   9|          9|      16|         65.0|          0|          1.0|
|      Charlie Batch|  2005|  10|         13|      19|        150.0|          0|          0.0|
|      Charlie Batch|  2005|  16|          1|       1|         31.0|          1|          0.0|
|         Jeff Blake|  2005|   2|          1|       1|         11.0|          0|          0.0|
|         Jeff Blake|  2005|  17|          7|       8|         44.0|          1|          0.0|
|       Drew Bledsoe|  2005|   1|         18|      24|        226.0|          3|          0.0|
|   

To calculate the sum and average of the statistics for each player and season combination, we also use `groupby()` and `agg()`. We can see that our original DataFrame of 11,750 records went down to 1,456 records.

In [57]:
#group DataFrame by player and season and calculate the sum and mean for each statistics
sql_df_stats = sql_short.groupBy(["player_display_name","season"]).agg(sum("week"), sum("completions"), sum("attempts"), sum("passing_yards"), sum("passing_tds"), sum("interceptions"), 
                                                        avg("week") , avg("completions"), avg("attempts"), avg("passing_yards"),avg("passing_tds"), avg("interceptions"))
#calculate and print out the number of records in the new DataFrame
print(f"There are {sql_df_stats.count()} unique player/season combinations.")
sql_df_stats.show()

There are 1456 unique player/season combinations.
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|player_display_name|season|sum(week)|sum(completions)|sum(attempts)|sum(passing_yards)|sum(passing_tds)|sum(interceptions)|         avg(week)|  avg(completions)|     avg(attempts)|avg(passing_yards)|  avg(passing_tds)|avg(interceptions)|
+-------------------+------+---------+----------------+-------------+------------------+----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|      Jake Delhomme|  2006|       99|             263|          431|            2805.0|              17|              11.0| 7.615384615384615| 20.23076923076923| 33.15384615384615|215.76923076923077|1.3076923076923077|0.846153846153

To make running actions on our DataFrame easier, we can use `withColumnRenamed` to rename the columns to names that are easier to work with.

In [58]:
#rename statistics columns
sql_df_stats = sql_df_stats.withColumnRenamed('sum(week)', 'sum_week').withColumnRenamed("sum(completions)", "sum_completions").withColumnRenamed("sum(attempts)","sum_attempts").withColumnRenamed("sum(passing_yards)","sum_passing_yards").withColumnRenamed("sum(passing_tds)","sum_passing_tds").withColumnRenamed("sum(interceptions)","sum_interceptions").withColumnRenamed("avg(week)","avg_week").withColumnRenamed("avg(completions)","avg_completions").withColumnRenamed("avg(attempts)","avg_attempts").withColumnRenamed("avg(passing_yards)","avg_passing_yards").withColumnRenamed("avg(passing_tds)","avg_passing_tds").withColumnRenamed("avg(interceptions)","avg_interceptions")
                

In [29]:
sql_df_stats.columns

['player_display_name',
 'season',
 'sum_week',
 'sum_completions',
 'sum_attempts',
 'sum_passing_yards',
 'sum_passing_tds',
 'sum_interceptions',
 'avg_week',
 'avg_completions',
 'avg_attempts',
 'avg_passing_yards',
 'avg_passing_tds',
 'avg_interceptions']

We now create new columns with `withColumn()` to calculate the completion percentage and touchdown/interception ratio and take a look at the first 3 rows.

In [59]:
#calculate and add completion_percentage variable
sql_df_stats = sql_df_stats.withColumn('completion_percentage', sql_df_stats.sum_completions / sql_df_stats.sum_attempts)
#calculate and add tds_int_ratio variable
sql_df_stats = sql_df_stats.withColumn('td_int_ratio', sql_df_stats.sum_passing_tds / sql_df_stats.sum_interceptions)
sql_df_stats.show(3)

+-------------------+------+--------+---------------+------------+-----------------+---------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum_week|sum_completions|sum_attempts|sum_passing_yards|sum_passing_tds|sum_interceptions|         avg_week|  avg_completions|     avg_attempts| avg_passing_yards|   avg_passing_tds| avg_interceptions|completion_percentage|      td_int_ratio|
+-------------------+------+--------+---------------+------------+-----------------+---------------+-----------------+-----------------+-----------------+-----------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|      99|            263|         431|           2805.0|             17|             11.0|7.615384615384615|20.23076923076923|33.15384615384615|215.76923076923

To subset rows where the number of attempts is at least 50, we use `where()` to set the condition on the `sum_attempts` column. We see that there are 966 records that meet this condition matching our results from pandas-on-Spark.

In [60]:
#subset DataFrame to include records where attempts is at least 50
sql_df_above_50 = sql_df_stats.where(sql_df_stats.sum_attempts >= 50) 
#calculate and print length of new DataFrame
print(f"There are {sql_df_above_50.count()} records where the number of attempts is at least 50.")
sql_df_above_50.show()

There are 966 records where the number of attempts is at least 50.
+-------------------+------+--------+---------------+------------+-----------------+---------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|player_display_name|season|sum_week|sum_completions|sum_attempts|sum_passing_yards|sum_passing_tds|sum_interceptions|          avg_week|   avg_completions|      avg_attempts| avg_passing_yards|   avg_passing_tds| avg_interceptions|completion_percentage|      td_int_ratio|
+-------------------+------+--------+---------------+------------+-----------------+---------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+---------------------+------------------+
|      Jake Delhomme|  2006|      99|            263|         431|           2805.0|             17|           

We now want to sort our DataFrame. First, we sort in a descending fashion by completion percentage with `sort()`. As we saw in the pandas-on-Spark section, there are a number of players that had a completion percentage of 100%.

In [61]:
#sort descending by completion_percentage
descending_cp = sql_df_stats.sort(sql_df_stats.completion_percentage, ascending = False)
#display only player_display_name, season, completions, attempts, and completion_percentage columns
descending_cp.select(['player_display_name','season','sum_completions','sum_attempts','completion_percentage']).show(40)

+-------------------+------+---------------+------------+---------------------+
|player_display_name|season|sum_completions|sum_attempts|completion_percentage|
+-------------------+------+---------------+------------+---------------------+
|     Anthony Wright|  2006|              3|           3|                  1.0|
|     Matt Gutierrez|  2007|              1|           1|                  1.0|
|     Matt Gutierrez|  2009|              1|           1|                  1.0|
|         Brad Smith|  2009|              1|           1|                  1.0|
|        Billy Volek|  2010|              1|           1|                  1.0|
|      Jordan Palmer|  2010|              3|           3|                  1.0|
|       Dennis Dixon|  2008|              1|           1|                  1.0|
|         Colt McCoy|  2013|              1|           1|                  1.0|
|       Tyrod Taylor|  2011|              1|           1|                  1.0|
|      Trent Edwards|  2012|            

We now want to sort the DataFrame in an ascending fashion based off the touchdown/interception ratio. Sorting the DataFrame shows a number of `NULL` values that were not appartent in the pandas-on-Spark version. **We see that in Spark SQL, dividing by 0 results in NULL which is considered lower than 0, so we see these values first. In pandas-on-spark, diving by 0 resutls in infinity and therefore these values are present at the end of the dataset when sorting in an ascending fashion.**

In [42]:
#sort ascending by td_int_ratio
ascending_td = sql_df_stats.sort(sql_df_stats.td_int_ratio)
#show
ascending_td.select(['player_display_name','season','sum_passing_tds', 'sum_interceptions', 'td_int_ratio']).show(40)

+-------------------+------+---------------+-----------------+------------+
|player_display_name|season|sum_passing_tds|sum_interceptions|td_int_ratio|
+-------------------+------+---------------+-----------------+------------+
|       Gus Frerotte|  2006|              0|              0.0|        NULL|
|        Josh McCown|  2009|              0|              0.0|        NULL|
|       Brad Johnson|  2007|              0|              0.0|        NULL|
|       Ingle Martin|  2006|              0|              0.0|        NULL|
| Charlie Whitehurst|  2006|              0|              0.0|        NULL|
|       Todd Collins|  2005|              0|              0.0|        NULL|
|     Tim Hasselbeck|  2007|              0|              0.0|        NULL|
|        A.J. Feeley|  2006|              3|              0.0|        NULL|
|         Craig Nall|  2007|              1|              0.0|        NULL|
|        Matt Schaub|  2005|              4|              0.0|        NULL|
|         Je

We can remove the `NULL` values from the DataFrame and now see 0 values that are not being divided by 0. 

In [45]:
ascending_td = sql_df_stats.where(isnotnull(sql_df_stats.td_int_ratio)).sort(sql_df_stats.td_int_ratio)
ascending_td.select(['player_display_name','season','sum_passing_tds', 'sum_interceptions', 'td_int_ratio']).show(40)

+-------------------+------+---------------+-----------------+------------+
|player_display_name|season|sum_passing_tds|sum_interceptions|td_int_ratio|
+-------------------+------+---------------+-----------------+------------+
|         Matt Flynn|  2009|              0|              1.0|         0.0|
| Charlie Whitehurst|  2015|              0|              1.0|         0.0|
|     Curtis Painter|  2009|              0|              2.0|         0.0|
|      Brodie Croyle|  2010|              0|              1.0|         0.0|
|      Tyler Thigpen|  2007|              0|              1.0|         0.0|
|          Jon Kitna|  2005|              0|              2.0|         0.0|
|     Richard Bartel|  2010|              0|              1.0|         0.0|
|   Brooks Bollinger|  2006|              0|              1.0|         0.0|
|       Brock Berlin|  2007|              0|              1.0|         0.0|
|           Joe Webb|  2010|              0|              3.0|         0.0|
|      Jorda

In this section, we used Spark SQL to look at our NFL dataset, subset the data based off certain conditions, create new statistics, sort our data, and make conclusions on what the data showed.